# Badanie Efektywności Reprezentacji Ortogonalnej Sygnałów 1D

---


## 1. Założenia Badawcze (Metoda Monte Carlo)
W tej części projektu odchodzimy od statycznego progu kompresji na rzecz badania **kosztu reprezentacji** sygnału. 
* **Próba:** Przeprowadzamy testy na $N=100$ niezależnych realizacjach dla każdej z 3 klas sygnałów (**APRBS**, **Multisine**, **Filtered Noise**).
* **Cel:** Wyznaczenie współczynnika $K$ – minimalnej liczby współczynników ortogonalnych niezbędnych do odtworzenia sygnału z zachowaniem **95% jego całkowitej energii**.
* **Hipoteza:** Wartość $K$ będzie zmienną losową zależną od charakterystyki sygnału. Spodziewamy się najniższych średnich wartości dla klasy *Multisine* (sygnały rzadkie w dziedzinie częstotliwości) i najwyższych dla *APRBS* (sygnały szerokopasmowe).

---

## 2. Podstawy Teoretyczne

### 2.1. Transformata Fouriera (Analiza)
Dyskretna Transformata Fouriera (DFT) pozwala na dekompozycję sygnału z dziedziny czasu na sumę ortogonalnych funkcji bazowych (zespolonych wykładniczych/sinusoidalnych).



Wzór na współczynniki widmowe:
$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-j \frac{2\pi}{N} kn}$$

Każdy współczynnik $X[k]$ reprezentuje amplitudę i fazę składowej o konkretnej częstotliwości.

### 2.2. Odwrotna Transformata Fouriera (Synteza)
Proces odwrotny pozwala na rekonstrukcję sygnału czasowego. W procesie kompresji, część współczynników $X[k]$ (te o najmniejszej energii) jest zerowana, co prowadzi do przybliżonej rekonstrukcji:

$$\hat{x}[n] = \frac{1}{N} \sum_{k=0}^{N-1} \hat{X}[k] \cdot e^{j \frac{2\pi}{N} kn}$$



### 2.3. Mierzenie Energii i Twierdzenie Parsevala
Energia sygnału jest miarą jego zawartości informacyjnej. Zgodnie z **Twierdzeniem Parsevala**, całkowita energia w dziedzinie czasu odpowiada energii w dziedzinie częstotliwości:

$$E_{total} = \sum_{n=0}^{N-1} |x[n]|^2 = \frac{1}{N} \sum_{k=0}^{N-1} |X[k]|^2$$

Aby wyznaczyć współczynnik $K$ dla progu 95%, stosujemy następującą procedurę:
1. Obliczamy moc poszczególnych składowych: $P[k] = |X[k]|^2$.
2. Sortujemy moce $P[k]$ malejąco: $P_{sorted}$.
3. Wyznaczamy najmniejsze $K$ spełniające nierówność:
$$\frac{\sum_{i=1}^{K} P_{sorted}[i]}{E_{total}} \geq 0.95$$



---

## 3. Algorytm Przetwarzania (Workflow)
Dla każdej z 100 iteracji w pętli realizujemy następujące kroki:
1. **Generowanie:** Tworzenie sygnału za pomocą klasy generatora z losowymi parametrami (częstotliwości, czasy utrzymania, szumy).
2. **Transformacja:** Wyznaczenie widma przy użyciu algorytmu FFT (`numpy.fft.rfft`).
3. **Analiza Energii:** Sortowanie energii współczynników i obliczenie sumy skumulowanej.
4. **Ekstrakcja K:** Zapisanie liczby współczynników $K$ niezbędnych do osiągnięcia progu 0.95 energii.
5. **Statystyka:** Wyznaczenie wartości średniej $\mu_K$ oraz odchylenia standardowego $\sigma_K$ dla każdej z grup sygnałów.

---
---
---


# KOD

## Biblioteki

In [ ]:
%matplotlib widget
# %matplotlib tk

from matplotlib import pyplot as plt
from DatasetCreator import DatasetCreator
from Plotter import Plotter 
from Fourier import Fourier
from EnergyAnalyzer import EnergyAnalyzer
import numpy as np

## Generacja sygnałów
Dodatkowo przykładowe wykresy

In [ ]:
creator = DatasetCreator(t_span=[0, 500], dt=0.1, amp_range=(3, 90), noise_level=0.0)
dataset = creator.create_dataset(n_aprbs=10, n_multisine=10, n_noise=10)

for category, signals in dataset.items():
    print(f"\n=== Kategoria: {category.upper()} ===")
    for i, traj in enumerate(signals[:1]):  # Pokazujemy tylko 1 przykład
        print(f"Trajektoria nr {i+1}:")
        print(f"  t (pierwsze 5): {traj.t[:5]}")
        print(f"  y (pierwsze 5): {traj.y[:5]}")
        print(f"  Długość wektora: {len(traj.y)} próbek")
        print("-" * 30)

# Rysowanie
# Chcę porównać pierwszy, środkowy i ostatni sygnał ze wszystkich kategorii 
Plotter.plot_indices(dataset['aprbs'], "aprbs", indices=[0, 5, 9])

Plotter.plot_indices(dataset['multisine'], "multisine", indices=[0, 5, 9])

Plotter.plot_indices(dataset['noise'], "noise", indices=[0, 5, 9])

## FOURIER

In [ ]:
# 1. Generujemy dane
creator = DatasetCreator(t_span=[0, 500], dt=0.1, amp_range=(3, 90), noise_level=0.0)
dataset = creator.create_dataset(n_aprbs=10, n_multisine=10, n_noise=10)

sample_signal = dataset['aprbs'][0]  # Pobieramy pierwszy sygnał z kategorii APRBS

# 2. Inicjalizacja klasy Fourier
fourier_engine = Fourier(sample_signal.t, sample_signal.y)

# 3. Obliczenie transformaty prostej dla zakresu częstotliwości
omega_limit = 5.0  # Zakres częstotliwości do analizy
omega_vec = np.linspace(-omega_limit, omega_limit, 500)
f_omega = fourier_engine.transform(omega_vec)

print(f"--- Transformaty dla omega w zakresie [-{omega_limit}, {omega_limit}] ---")
print(f"Omega  (pierwsze 5): {f_omega[:5]}")

# 4. Obliczenie transformaty odwrotnej (rekonstrukcja sygnału)
# Używamy mniejszego num_omega dla szybkości lub większego dla precyzji
y_reconstructed = fourier_engine.inverse_transform(
    sample_signal.t, 
    omega_limit=omega_limit, 
    num_omega=1000
)

# 5. Wizualizacja wyników
# Wykres widma (Część rzeczywista, urojona i moduł)
Plotter.plot_fourier_transform(omega_vec, f_omega)

# Wykres porównawczy (Oryginał vs Rekonstrukcja)
Plotter.plot_fourier_comparison(
    sample_signal.t, 
    sample_signal.y, 
    y_reconstructed, 
    omega=omega_limit
)


## ANALIZA ENERGII

In [9]:
# 1. Generujemy dane
creator = DatasetCreator(t_span=[0, 500], dt=0.1, amp_range=(3, 90), noise_level=0.0)
dataset = creator.create_dataset(n_aprbs=10, n_multisine=10, n_noise=10)

print("--- ANALIZA SYGNAŁU MULTISINE ---")# Pobieramy pierwszy sygnał z listy
signal = dataset['multisine'][0] 

# 2. Analiza energii w dziedzinie czasu
e_time = EnergyAnalyzer.calculate_total_energy(signal)
print(f"Energia w dziedzinie czasu: {e_time:.2f}")

# 3. Analiza energii w dziedzinie częstotliwości (FFT)
coeffs = np.fft.rfft(signal.y)

results = EnergyAnalyzer.analyze_energy_distribution(coeffs, N=len(signal.y), threshold=0.95)
print(f"Wspołczynniki FFT - liczba: {len(coeffs)}...")  # Pokazujemy tylko pierwsze 5 współczynników
print(f"Wspołczynniki FFT: {coeffs[:5]}...")  # Pokazujemy tylko pierwsze 5 współczynników
print(f"Energia w dziedzinie częstotliwości: {results['total_energy_freq']:.2f}")
print(f"Liczba współczynników (K) dla 95% energii: {results['k_threshold']}")

print("\nSygnał aprbs")
signal = dataset['aprbs'][0] 
# 2. Analiza energii w dziedzinie czasu
e_time = EnergyAnalyzer.calculate_total_energy(signal)
print(f"Energia w dziedzinie czasu: {e_time:.2f}")

# 3. Analiza energii w dziedzinie częstotliwości (FFT)
# Zakładając, że Twoja klasa Fourier używa np.fft.rfft:
coeffs = np.fft.rfft(signal.y)

results = EnergyAnalyzer.analyze_energy_distribution(coeffs, N=len(signal.y), threshold=0.95)
print(f"Wspołczynniki FFT - liczba: {len(coeffs)}...")  # Pokazujemy tylko pierwsze 5 współczynników
print(f"Wspołczynniki FFT: {coeffs[:5]}...")  # Pokazujemy tylko pierwsze 5 współczynników
print(f"Energia w dziedzinie częstotliwości: {results['total_energy_freq']:.2f}")
print(f"Liczba współczynników (K) dla 95% energii: {results['k_threshold']}")

--- Start generowania zestawu danych (Noise Level: 0.0) ---


Generowanie NOISE: 100%|██████████| 10/10 [00:00<00:00, 313.35it/s]


--- Generowanie zakończone ---
Kategoria APRBS: 10 sygnałów
Kategoria MULTISINE: 10 sygnałów
Kategoria NOISE: 10 sygnałów
--- ANALIZA SYGNAŁU MULTISINE ---
Energia w dziedzinie czasu: 10071324.91
Wspołczynniki FFT - liczba: 2501...
Wspołczynniki FFT: [222610.3201187    +0.j         -10989.3592465 +4154.26314732j
  -3979.67057651-2599.94318019j   2937.49838376+3630.3159325j
   1652.47490628+8088.06735599j]...
Energia w dziedzinie częstotliwości: 9991197.94
Liczba współczynników (K) dla 95% energii: 1

Sygnał aprbs
Energia w dziedzinie czasu: 16558537.43
Wspołczynniki FFT - liczba: 2501...
Wspołczynniki FFT: [246675.30570039    +0.j          20953.11563556+35548.67090972j
 -13993.86809249+18210.36013231j  -3924.54995522 -7327.12873249j
  22953.33563777+28886.88142732j]...
Energia w dziedzinie częstotliwości: 14364142.30
Liczba współczynników (K) dla 95% energii: 10


## COŚ

In [ ]:
import numpy as np
import scipy.fftpack as fftpack
import matplotlib.pyplot as plt

# 1. Tworzymy sygnał testowy (suma sinusów - model prostego dźwięku)
N = 512
t = np.linspace(0, 1, N)
signal = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 20 * t) + 0.2 * np.random.randn(N)

# 2. Obliczamy współczynniki w dwóch bazach
# FFT (Fourier) - bierzemy moduł, bo FFT daje liczby zespolone
coef_fft = np.abs(np.fft.rfft(signal)) 
# DCT (Kosinusowa) - standard w audio
coef_dct = np.abs(fftpack.dct(signal, norm='ortho'))

# 3. Funkcja do obliczania skumulowanej energii (zbieżności)
def get_energy_convergence(coeffs):
    # Sortujemy współczynniki od największego (najważniejszego) do najmniejszego
    sorted_coeffs = np.sort(np.abs(coeffs))[::-1]
    # Liczymy sumę skumulowaną kwadratów (energia to kwadrat amplitudy)
    energy_cumulative = np.cumsum(sorted_coeffs**2)
    # Normalizujemy do 1.0 (czyli do 100%)
    return energy_cumulative / energy_cumulative[-1]

# 4. Obliczamy krzywe zbieżności
energy_fft = get_energy_convergence(coef_fft)
energy_dct = get_energy_convergence(coef_dct)

# 5. Wykres porównawczy
plt.figure(figsize=(10, 5))
plt.plot(energy_dct[:100], label='Baza DCT (Kosinusowa)', linewidth=2)
plt.plot(energy_fft[:100], label='Baza FFT (Fouriera)', linestyle='--')
plt.axhline(y=0.95, color='r', alpha=0.3, label='Próg 95% energii')
plt.title("Porównanie zbieżności energii (pierwsze 100 współczynników)")
plt.xlabel("Liczba użytych współczynników")
plt.ylabel("Ułamek całkowitej energii")
plt.legend()
plt.grid(True)
plt.show()